<a href="https://colab.research.google.com/github/abiodunadesesan/FlyRank-ml-Internship-/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abiodunadesesan/FlyRank-ml-Internship-/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [2]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

### Research Question

Which visible pages have the greatest opportunity for CTR improvement, and how can we rank them for content optimization?

### Decision

Which webpages should be prioritized for CTR optimization?

### Who will use the result?

Content and SEO teams can use the ranking to decide which pages to review first for possible improvements to titles, meta descriptions, content, and other on-page elements.

### Objective

Build a repeatable CTR opportunity scoring approach using the FlyRank search dataset, compare it with a transparent baseline, validate the result honestly, and turn the analysis into a ranked action playbook.

In [3]:
import duckdb
import pandas as pd

print("DuckDB:", duckdb.__version__)
print("Pandas:", pd.__version__)

DuckDB: 1.3.2
Pandas: 2.2.3


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [4]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("FlyRank warehouse authentication is ready.")

FlyRank warehouse authentication is ready.


In [5]:
# Inspect the available FlyRank warehouse tables

tables = con.sql("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
ORDER BY table_name
""").df()

tables

,table_name


In [6]:
# Check whether the Hugging Face secret is available
print("Token available:", bool(HF_TOKEN))

Token available: True


In [7]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("FlyRank warehouse authentication is ready.")

FlyRank warehouse authentication is ready.


In [8]:
rel = "hf://datasets/FlyRank/internship-warehouse"

test = con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""").df()

test

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows
0,78835655


In [9]:
schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""").df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [10]:
summary = con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS contents,
    COUNT(DISTINCT CONCAT(
        CAST(report_date AS VARCHAR), '|',
        client_hash_id, '|',
        content_hash_id
    )) AS unique_grain_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""").df()

summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows,clients,contents,unique_grain_rows
0,2025-01-27,2026-06-30,78835655,70,427292,78829265


In [11]:
duplicates = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 20
""").df()

duplicates

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count
0,2026-06-15,client_a2eeb8899886adde,content_9ce5ba9ac4527940,2
1,2026-06-17,client_1a730cb2640a1abf,content_965b9031838c130f,2
2,2026-06-17,client_1a730cb2640a1abf,content_5c5b96cd4d1829d8,2
3,2026-06-14,client_aef6ffea193da149,content_608ff8abdfba1469,2
4,2026-06-18,client_def0955f7a377868,content_40c868cfcb49537c,2
5,2026-06-20,client_1a730cb2640a1abf,content_6af788521680129d,2
6,2026-06-19,client_06d356715a8ff3b6,content_6c6a2658f025ec02,2
7,2026-06-21,client_1a8bf67cad4ee525,content_567e7aa5ce2a36e6,2
8,2026-06-22,client_8ddc46da5414ffd8,content_55335cfcf3499724,2
9,2026-06-23,client_1a8bf67cad4ee525,content_e85eff1aad797f4c,2


In [12]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

print("FlyRank warehouse connection restored.")

FlyRank warehouse connection restored.


In [13]:
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)

Pandas: 2.2.3


In [14]:
import pandas as pd
import numpy as np
import duckdb
from google.colab import userdata

# Restore authentication
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

# Restore DuckDB connection
con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

# Restore warehouse location
rel = "hf://datasets/FlyRank/internship-warehouse"

# Restore time windows
TRAIN_START = "2025-10-01"
TRAIN_END   = "2026-03-31"

VALID_START = "2026-04-01"
VALID_END   = "2026-06-30"

print("Capstone state restored.")
print("Training:", TRAIN_START, "to", TRAIN_END)
print("Validation:", VALID_START, "to", VALID_END)

Capstone state restored.
Training: 2025-10-01 to 2026-03-31
Validation: 2026-04-01 to 2026-06-30


In [15]:
page_features = con.sql(f"""
WITH clean_data AS (
    SELECT DISTINCT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE gsc_data_available = TRUE
      AND gsc_impressions > 0
),

page_features AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,

        SUM(gsc_impressions * gsc_avg_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS avg_position,

        COUNT(DISTINCT report_date) AS active_days

    FROM clean_data

    WHERE report_date BETWEEN '{TRAIN_START}' AND '{TRAIN_END}'

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM page_features
WHERE impressions >= 100
""").df()

print("Training pages:", len(page_features))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training pages: 126622


In [16]:
validation_outcomes = con.sql(f"""
WITH clean_data AS (
    SELECT DISTINCT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE gsc_data_available = TRUE
      AND gsc_impressions > 0
),

future_pages AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS validation_impressions,
        SUM(gsc_clicks) AS validation_clicks,

        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS validation_ctr,

        SUM(gsc_impressions * gsc_avg_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS validation_avg_position,

        COUNT(DISTINCT report_date) AS validation_active_days

    FROM clean_data

    WHERE report_date BETWEEN '{VALID_START}' AND '{VALID_END}'

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM future_pages
WHERE validation_impressions >= 100
""").df()

print("Validation pages:", len(validation_outcomes))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Validation pages: 151996


In [17]:
model_dataset = page_features.merge(
    validation_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Pages in both periods:", len(model_dataset))

Pages in both periods: 104173


In [18]:
model_dataset["position_bucket"] = pd.cut(
    model_dataset["avg_position"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=["1-3", "4-5", "6-10", "11-20", "21+"],
    right=True
)

print(model_dataset["position_bucket"].value_counts().sort_index())

position_bucket
1-3      10519
4-5      17472
6-10     33221
11-20    21735
21+      21226
Name: count, dtype: int64


In [19]:
baseline_ctr = {
    "1-3": 0.005443,
    "4-5": 0.004597,
    "6-10": 0.003267,
    "11-20": 0.003472,
    "21+": 0.001559
}

model_dataset["baseline_ctr"] = (
    model_dataset["position_bucket"]
    .astype(str)
    .map(baseline_ctr)
)

print(
    "Pages with baseline CTR:",
    model_dataset["baseline_ctr"].notna().sum()
)

model_dataset[
    ["position_bucket", "ctr", "baseline_ctr"]
].head(10)

Pages with baseline CTR: 104173


,position_bucket,ctr,baseline_ctr
0,6-10,0.000934,0.003267
1,4-5,0.003347,0.004597
2,4-5,0.009184,0.004597
3,11-20,0.003414,0.003472
4,6-10,0.001468,0.003267
5,6-10,0.004228,0.003267
6,6-10,0.003376,0.003267
7,4-5,0.008086,0.004597
8,6-10,0.008035,0.003267
9,11-20,0.000000,0.003472


In [20]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

print("FlyRank warehouse connection restored.")

FlyRank warehouse connection restored.


In [21]:
import pandas as pd
import numpy as np
import duckdb
from google.colab import userdata

# -----------------------------
# 1. Restore connection
# -----------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

TRAIN_START = "2025-10-01"
TRAIN_END   = "2026-03-31"

VALID_START = "2026-04-01"
VALID_END   = "2026-06-30"


# -----------------------------
# 2. Build training features
# -----------------------------

page_features = con.sql(f"""
WITH clean_data AS (
    SELECT DISTINCT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE gsc_data_available = TRUE
      AND gsc_impressions > 0
)

SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,

    SUM(gsc_clicks) * 1.0
        / NULLIF(SUM(gsc_impressions), 0) AS ctr,

    SUM(gsc_impressions * gsc_avg_position) * 1.0
        / NULLIF(SUM(gsc_impressions), 0) AS avg_position,

    COUNT(DISTINCT report_date) AS active_days

FROM clean_data

WHERE report_date BETWEEN '{TRAIN_START}' AND '{TRAIN_END}'

GROUP BY
    client_hash_id,
    content_hash_id

HAVING SUM(gsc_impressions) >= 100
""").df()


# -----------------------------
# 3. Build future outcomes
# -----------------------------

validation_outcomes = con.sql(f"""
WITH clean_data AS (
    SELECT DISTINCT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE gsc_data_available = TRUE
      AND gsc_impressions > 0
)

SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS validation_impressions,
    SUM(gsc_clicks) AS validation_clicks,

    SUM(gsc_clicks) * 1.0
        / NULLIF(SUM(gsc_impressions), 0) AS validation_ctr,

    SUM(gsc_impressions * gsc_avg_position) * 1.0
        / NULLIF(SUM(gsc_impressions), 0) AS validation_avg_position,

    COUNT(DISTINCT report_date) AS validation_active_days

FROM clean_data

WHERE report_date BETWEEN '{VALID_START}' AND '{VALID_END}'

GROUP BY
    client_hash_id,
    content_hash_id

HAVING SUM(gsc_impressions) >= 100
""").df()


# -----------------------------
# 4. Join historical + future
# -----------------------------

model_dataset = page_features.merge(
    validation_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)


# -----------------------------
# 5. Position buckets
# -----------------------------

model_dataset["position_bucket"] = pd.cut(
    model_dataset["avg_position"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=["1-3", "4-5", "6-10", "11-20", "21+"],
    right=True
)


# -----------------------------
# 6. Historical CTR baseline
# -----------------------------

baseline_ctr = {
    "1-3": 0.005443,
    "4-5": 0.004597,
    "6-10": 0.003267,
    "11-20": 0.003472,
    "21+": 0.001559
}

model_dataset["baseline_ctr"] = (
    model_dataset["position_bucket"]
    .astype(str)
    .map(baseline_ctr)
)


# -----------------------------
# 7. Future CTR opportunity
# -----------------------------

model_dataset["ctr_gap"] = (
    model_dataset["baseline_ctr"]
    - model_dataset["validation_ctr"]
)

model_dataset["potential_clicks"] = (
    model_dataset["ctr_gap"]
    * model_dataset["validation_impressions"]
)


# -----------------------------
# 8. Verify
# -----------------------------

print("Training pages:", len(page_features))
print("Validation pages:", len(validation_outcomes))
print("Pages in both periods:", len(model_dataset))
print(
    "Pages with positive CTR gap:",
    (model_dataset["ctr_gap"] > 0).sum()
)
print(
    "Pages with zero/negative CTR gap:",
    (model_dataset["ctr_gap"] <= 0).sum()
)

print("\nSample:")
display(
    model_dataset[
        [
            "position_bucket",
            "baseline_ctr",
            "validation_ctr",
            "ctr_gap",
            "validation_impressions",
            "potential_clicks"
        ]
    ].head(10)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training pages: 126622
Validation pages: 151996
Pages in both periods: 104173
Pages with positive CTR gap: 77950
Pages with zero/negative CTR gap: 26223

Sample:


,position_bucket,baseline_ctr,validation_ctr,ctr_gap,validation_impressions,potential_clicks
0,11-20,0.003472,0.000000,0.003472,366.0,1.270752
1,21+,0.001559,0.004104,-0.002545,731.0,-1.860371
2,21+,0.001559,0.002037,-0.000478,1964.0,-0.938124
3,21+,0.001559,0.001241,0.000318,1611.0,0.511549
4,21+,0.001559,0.013436,-0.011877,1042.0,-12.375522
5,21+,0.001559,0.000000,0.001559,427.0,0.665693
6,11-20,0.003472,0.003401,0.000071,294.0,0.020768
7,21+,0.001559,0.002288,-0.000729,1748.0,-1.274868
8,11-20,0.003472,0.009153,-0.005681,437.0,-2.482736
9,6-10,0.003267,0.004213,-0.000946,712.0,-0.673896


In [22]:
print("model_dataset:", len(model_dataset))
print("page_features:", len(page_features))
print("validation_outcomes:", len(validation_outcomes))
print("opportunity_score exists:", "opportunity_score" in model_dataset.columns)

model_dataset: 104173
page_features: 126622
validation_outcomes: 151996
opportunity_score exists: False


In [23]:
import pandas as pd
import numpy as np

# Create historical CTR gap
model_dataset["historical_ctr_gap"] = (
    model_dataset["baseline_ctr"]
    - model_dataset["ctr"]
)

# Create opportunity score
model_dataset["opportunity_score"] = (
    model_dataset["historical_ctr_gap"].clip(lower=0)
    * np.log1p(model_dataset["impressions"])
)

print("Opportunity score created:", "opportunity_score" in model_dataset.columns)
print(model_dataset["opportunity_score"].describe())

Opportunity score created: True
count    104173.000000
mean          0.012219
std           0.011333
min           0.000000
25%           0.000000
50%           0.010094
75%           0.020595
max           0.069787
Name: opportunity_score, dtype: float64


In [24]:
top_opportunities = (
    model_dataset
    .sort_values("opportunity_score", ascending=False)
    [[
        "client_hash_id",
        "content_hash_id",
        "position_bucket",
        "ctr",
        "baseline_ctr",
        "historical_ctr_gap",
        "impressions",
        "active_days",
        "opportunity_score"
    ]]
    .head(20)
)

top_opportunities

,client_hash_id,content_hash_id,position_bucket,ctr,baseline_ctr,historical_ctr_gap,impressions,active_days,opportunity_score
57129,client_73cda7b4e4f265ea,content_fec55986a1868d62,1-3,0.000003,0.005443,0.005440,372381.0,182,0.069787
5107,client_73cda7b4e4f265ea,content_8e1334d6356668e3,1-3,0.000011,0.005443,0.005432,366197.0,182,0.069590
10565,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,1-3,0.000007,0.005443,0.005436,279673.0,135,0.068173
87219,client_23a62021009f63c4,content_44f34c0a90047651,1-3,0.000132,0.005443,0.005311,302868.0,65,0.067030
77007,client_e547b89c05043229,content_306bc78dff1eb683,1-3,0.000504,0.005443,0.004939,242280.0,135,0.061239
34764,client_23a62021009f63c4,content_2ac8c7995de53cd1,1-3,0.000191,0.005443,0.005252,94198.0,78,0.060151
77010,client_e547b89c05043229,content_c46df0fa61530d86,1-3,0.000487,0.005443,0.004956,170379.0,135,0.059697
23825,client_e547b89c05043229,content_35f10273f0baf406,1-3,0.000214,0.005443,0.005229,79593.0,135,0.059012
47678,client_23a62021009f63c4,content_bf078007df823490,1-3,0.000000,0.005443,0.005443,44707.0,20,0.058283
8418,client_23a62021009f63c4,content_66d3e7a515e4ec68,1-3,0.000132,0.005443,0.005311,53174.0,171,0.057795


In [25]:
print("CTR gap statistics")
print(model_dataset["ctr_gap"].describe())

print("\nPotential click statistics")
print(model_dataset["potential_clicks"].describe())

print("\nLargest positive opportunities")
display(
    model_dataset[
        model_dataset["potential_clicks"] > 0
    ][
        [
            "position_bucket",
            "baseline_ctr",
            "validation_ctr",
            "ctr_gap",
            "validation_impressions",
            "potential_clicks"
        ]
    ]
    .sort_values("potential_clicks", ascending=False)
    .head(20)
)

CTR gap statistics
count    104173.000000
mean          0.000941
std           0.003541
min          -0.101081
25%          -0.000017
50%           0.001559
75%           0.003267
max           0.005443
Name: ctr_gap, dtype: float64

Potential click statistics
count    104173.000000
mean          4.184639
std          60.768383
min       -4642.451430
25%          -0.020496
50%           1.138752
75%           4.835160
max        2294.267774
Name: potential_clicks, dtype: float64

Largest positive opportunities


,position_bucket,baseline_ctr,validation_ctr,ctr_gap,validation_impressions,potential_clicks
51268,1-3,0.005443,0.001103,0.004340,528618.0,2294.267774
82336,4-5,0.004597,0.001254,0.003343,611708.0,2045.021676
35819,4-5,0.004597,0.000909,0.003688,508071.0,1873.602387
75116,4-5,0.004597,0.002648,0.001949,908095.0,1769.512715
30219,4-5,0.004597,0.001299,0.003298,523610.0,1727.035170
76902,4-5,0.004597,0.001719,0.002878,596157.0,1715.533729
72581,6-10,0.003267,0.000224,0.003043,557202.0,1695.378934
24828,1-3,0.005443,0.002543,0.002900,565913.0,1641.264459
25300,4-5,0.004597,0.001515,0.003082,504799.0,1555.561003
43369,4-5,0.004597,0.001041,0.003556,413999.0,1472.153403


In [26]:
# Build a transparent historical CTR opportunity score

# Historical CTR gap
model_dataset["historical_ctr_gap"] = (
    model_dataset["baseline_ctr"] - model_dataset["ctr"]
)

# Only positive gaps represent potential opportunity
model_dataset["positive_ctr_gap"] = (
    model_dataset["historical_ctr_gap"].clip(lower=0)
)

# Log exposure reduces the dominance of extremely high-impression pages
model_dataset["exposure_score"] = np.log1p(
    model_dataset["impressions"]
)

# Stability rewards pages with more observed active days
model_dataset["stability_score"] = (
    model_dataset["active_days"] / 182
).clip(upper=1)

# Transparent action score
model_dataset["opportunity_score"] = (
    model_dataset["positive_ctr_gap"]
    * model_dataset["exposure_score"]
    * model_dataset["stability_score"]
)

print(model_dataset["opportunity_score"].describe())

print("\nTop opportunities based only on historical information:")

display(
    model_dataset[
        [
            "client_hash_id",
            "content_hash_id",
            "position_bucket",
            "ctr",
            "baseline_ctr",
            "historical_ctr_gap",
            "impressions",
            "active_days",
            "opportunity_score"
        ]
    ]
    .sort_values("opportunity_score", ascending=False)
    .head(20)
)

count    104173.000000
mean          0.007450
std           0.009105
min           0.000000
25%           0.000000
50%           0.003826
75%           0.011751
max           0.069787
Name: opportunity_score, dtype: float64

Top opportunities based only on historical information:


,client_hash_id,content_hash_id,position_bucket,ctr,baseline_ctr,historical_ctr_gap,impressions,active_days,opportunity_score
57129,client_73cda7b4e4f265ea,content_fec55986a1868d62,1-3,0.000003,0.005443,0.005440,372381.0,182,0.069787
5107,client_73cda7b4e4f265ea,content_8e1334d6356668e3,1-3,0.000011,0.005443,0.005432,366197.0,182,0.069590
15522,client_73cda7b4e4f265ea,content_1cb7263083e97ba1,1-3,0.000095,0.005443,0.005348,31425.0,182,0.055376
57128,client_73cda7b4e4f265ea,content_c9f840183215651b,4-5,0.000000,0.004597,0.004597,151750.0,182,0.054842
69606,client_73cda7b4e4f265ea,content_254500f1d708cd6b,1-3,0.000264,0.005443,0.005179,37878.0,182,0.054598
2594,client_23a62021009f63c4,content_dcc8191464a7e5b0,1-3,0.000109,0.005443,0.005334,27433.0,182,0.054507
8418,client_23a62021009f63c4,content_66d3e7a515e4ec68,1-3,0.000132,0.005443,0.005311,53174.0,171,0.054302
10879,client_73cda7b4e4f265ea,content_f57e4e8c0208f271,1-3,0.000405,0.005443,0.005038,46892.0,182,0.054185
10886,client_73cda7b4e4f265ea,content_97d7732bcbfc6931,1-3,0.000603,0.005443,0.004840,68038.0,182,0.053863
65056,client_fef1a8f436438636,content_5cc6fa5852bf25f1,1-3,0.000274,0.005443,0.005169,32853.0,182,0.053757


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Method

The analysis uses page-level historical search performance aggregated over the training period from 2025-10-01 to 2026-03-31.

The unit of analysis is a client-content page pair.

Historical features include impressions, clicks, CTR, average position, active days, and position bucket.

The position bucket provides a transparent baseline CTR based on observed CTR within each position range.

### Label

The validation period is 2026-04-01 to 2026-06-30.

The main outcome is the CTR gap:

CTR gap = baseline CTR - validation CTR

A positive CTR gap means the page achieved a lower CTR during validation than the position-based benchmark.

### Baseline

The baseline uses observed CTR by position bucket:

- 1-3: 0.5443%
- 4-5: 0.4597%
- 6-10: 0.3267%
- 11-20: 0.3472%
- 21+: 0.1559%

### Validation design

A time-aware split is used.

Training:
2025-10-01 to 2026-03-31

Validation:
2026-04-01 to 2026-06-30

The opportunity score is calculated using historical information only. Validation-period outcomes are held out until evaluation.

### Leakage check

Future validation CTR, clicks, impressions, position, and other validation outcomes are not used to construct the historical opportunity score.

The validation period is used only to evaluate whether pages ranked as opportunities subsequently show a positive CTR gap.

### Interpretation

The results are directional and intended for decision support. They identify pages that may deserve human review for CTR improvement. They do not establish causal effects or prove how Google's ranking system works.

In [27]:
# Restore capstone period definitions

TRAIN_START = "2025-10-01"
TRAIN_END = "2026-03-31"

VALID_START = "2026-04-01"
VALID_END = "2026-06-30"

print("Training:", TRAIN_START, "to", TRAIN_END)
print("Validation:", VALID_START, "to", VALID_END)

print("model_dataset exists:", "model_dataset" in globals())

if "model_dataset" in globals():
    print("Rows:", len(model_dataset))
    print("Opportunity score exists:",
          "opportunity_score" in model_dataset.columns)

Training: 2025-10-01 to 2026-03-31
Validation: 2026-04-01 to 2026-06-30
model_dataset exists: True
Rows: 104173
Opportunity score exists: True


In [28]:
# Check that the historical opportunity score is ready for evaluation

print("Rows:", len(model_dataset))
print("Opportunity score available:",
      "opportunity_score" in model_dataset.columns)

print("\nOpportunity score statistics:")
print(model_dataset["opportunity_score"].describe())

Rows: 104173
Opportunity score available: True

Opportunity score statistics:
count    104173.000000
mean          0.007450
std           0.009105
min           0.000000
25%           0.000000
50%           0.003826
75%           0.011751
max           0.069787
Name: opportunity_score, dtype: float64


In [29]:
top_opportunities = (
    model_dataset
    .sort_values("opportunity_score", ascending=False)
    [[
        "client_hash_id",
        "content_hash_id",
        "position_bucket",
        "ctr",
        "baseline_ctr",
        "historical_ctr_gap",
        "impressions",
        "active_days",
        "opportunity_score"
    ]]
    .head(20)
)

top_opportunities

,client_hash_id,content_hash_id,position_bucket,ctr,baseline_ctr,historical_ctr_gap,impressions,active_days,opportunity_score
57129,client_73cda7b4e4f265ea,content_fec55986a1868d62,1-3,0.000003,0.005443,0.005440,372381.0,182,0.069787
5107,client_73cda7b4e4f265ea,content_8e1334d6356668e3,1-3,0.000011,0.005443,0.005432,366197.0,182,0.069590
15522,client_73cda7b4e4f265ea,content_1cb7263083e97ba1,1-3,0.000095,0.005443,0.005348,31425.0,182,0.055376
57128,client_73cda7b4e4f265ea,content_c9f840183215651b,4-5,0.000000,0.004597,0.004597,151750.0,182,0.054842
69606,client_73cda7b4e4f265ea,content_254500f1d708cd6b,1-3,0.000264,0.005443,0.005179,37878.0,182,0.054598
2594,client_23a62021009f63c4,content_dcc8191464a7e5b0,1-3,0.000109,0.005443,0.005334,27433.0,182,0.054507
8418,client_23a62021009f63c4,content_66d3e7a515e4ec68,1-3,0.000132,0.005443,0.005311,53174.0,171,0.054302
10879,client_73cda7b4e4f265ea,content_f57e4e8c0208f271,1-3,0.000405,0.005443,0.005038,46892.0,182,0.054185
10886,client_73cda7b4e4f265ea,content_97d7732bcbfc6931,1-3,0.000603,0.005443,0.004840,68038.0,182,0.053863
65056,client_fef1a8f436438636,content_5cc6fa5852bf25f1,1-3,0.000274,0.005443,0.005169,32853.0,182,0.053757


In [30]:
import pandas as pd
import numpy as np

# Recreate historical CTR gap
model_dataset["historical_ctr_gap"] = (
    model_dataset["baseline_ctr"] - model_dataset["ctr"]
)

# Recreate opportunity score
model_dataset["opportunity_score"] = (
    model_dataset["historical_ctr_gap"].clip(lower=0)
    * np.log1p(model_dataset["impressions"])
)

# Create score deciles
model_dataset["score_decile"] = pd.qcut(
    model_dataset["opportunity_score"],
    q=10,
    labels=False,
    duplicates="drop"
)

print("Model rows:", len(model_dataset))
print("Opportunity score exists:",
      "opportunity_score" in model_dataset.columns)
print("Score deciles created:",
      "score_decile" in model_dataset.columns)

print("\nOpportunity score statistics:")
print(model_dataset["opportunity_score"].describe())

Model rows: 104173
Opportunity score exists: True
Score deciles created: True

Opportunity score statistics:
count    104173.000000
mean          0.012219
std           0.011333
min           0.000000
25%           0.000000
50%           0.010094
75%           0.020595
max           0.069787
Name: opportunity_score, dtype: float64


In [31]:
decile_results = (
    model_dataset
    .groupby("score_decile", observed=True)
    .agg(
        pages=("opportunity_score", "size"),
        avg_score=("opportunity_score", "mean"),
        avg_validation_ctr_gap=("ctr_gap", "mean"),
        median_validation_ctr_gap=("ctr_gap", "median"),
        total_validation_impressions=("validation_impressions", "sum"),
        total_potential_clicks=("potential_clicks", "sum")
    )
    .reset_index()
    .sort_values("score_decile")
)

decile_results

,score_decile,pages,avg_score,avg_validation_ctr_gap,median_validation_ctr_gap,total_validation_impressions,total_potential_clicks
0,0,31252,0.000036,-0.001159,-0.000103,254964155.0,-352798.432786
1,1,10417,0.004803,0.000915,0.001388,63770977.0,54793.253877
2,2,10418,0.008836,0.000955,0.001559,34874032.0,36651.156420
3,3,10417,0.012050,0.001470,0.001559,60741481.0,89166.038698
4,4,10417,0.016922,0.001772,0.002522,47879561.0,89918.315593
5,5,10417,0.020607,0.002156,0.002870,46017383.0,101827.292039
6,6,10417,0.024551,0.002283,0.002772,69910377.0,124922.767070
7,7,10418,0.034308,0.003338,0.003608,87491159.0,291446.009266


In [32]:
# Validation lift from the lowest to highest opportunity-score decile

low_decile = decile_results.iloc[0]
high_decile = decile_results.iloc[-1]

low_gap = low_decile["avg_validation_ctr_gap"]
high_gap = high_decile["avg_validation_ctr_gap"]

print("Lowest decile validation CTR gap:", low_gap)
print("Highest decile validation CTR gap:", high_gap)

print(
    "Absolute lift:",
    high_gap - low_gap
)

if low_gap != 0:
    print(
        "Ratio of highest to lowest:",
        high_gap / low_gap
    )

Lowest decile validation CTR gap: -0.0011593434608047244
Highest decile validation CTR gap: 0.0033379639263265725
Absolute lift: 0.0044973073871312965
Ratio of highest to lowest: -2.879184675790229


In [33]:
# Check whether validation CTR gap generally increases with score decile

correlation = decile_results[
    ["score_decile", "avg_validation_ctr_gap"]
].corr().iloc[0, 1]

print("Decile-to-validation-gap correlation:", correlation)

Decile-to-validation-gap correlation: 0.9327716423818802


In [34]:
# Build ranked action queue

action_queue = model_dataset.copy()

# Rank only pages with positive historical opportunity
action_queue = action_queue[
    action_queue["opportunity_score"] > 0
].copy()

# Reason codes
action_queue["reason_code"] = np.select(
    [
        (action_queue["position_bucket"].isin(["1-3", "4-5"])) &
        (action_queue["ctr"] < action_queue["baseline_ctr"]),

        (action_queue["position_bucket"] == "6-10") &
        (action_queue["ctr"] < action_queue["baseline_ctr"]),

        (action_queue["position_bucket"].isin(["11-20", "21+"])) &
        (action_queue["ctr"] < action_queue["baseline_ctr"])
    ],
    [
        "HIGH_VISIBILITY_CTR_GAP",
        "MID_VISIBILITY_CTR_GAP",
        "LOW_VISIBILITY_CTR_GAP"
    ],
    default="CTR_REVIEW"
)

# Recommended action
action_queue["recommended_action"] = np.select(
    [
        action_queue["position_bucket"].isin(["1-3", "4-5"]),
        action_queue["position_bucket"] == "6-10",
        action_queue["position_bucket"].isin(["11-20", "21+"])
    ],
    [
        "Review title and meta description",
        "Review snippet and on-page relevance",
        "Review content relevance and search intent"
    ],
    default="Human review"
)

# Rank by opportunity score
action_queue = action_queue.sort_values(
    "opportunity_score",
    ascending=False
).reset_index(drop=True)

action_queue["rank"] = np.arange(1, len(action_queue) + 1)

# Keep the fields needed for the paper
action_queue = action_queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "position_bucket",
        "ctr",
        "baseline_ctr",
        "historical_ctr_gap",
        "impressions",
        "active_days",
        "opportunity_score",
        "reason_code",
        "recommended_action"
    ]
]

print("Ranked action queue:", len(action_queue))
action_queue.head(20)

Ranked action queue: 74675


,rank,client_hash_id,content_hash_id,position_bucket,ctr,baseline_ctr,historical_ctr_gap,impressions,active_days,opportunity_score,reason_code,recommended_action
0,1,client_73cda7b4e4f265ea,content_fec55986a1868d62,1-3,0.000003,0.005443,0.005440,372381.0,182,0.069787,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,1-3,0.000011,0.005443,0.005432,366197.0,182,0.069590,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
2,3,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,1-3,0.000007,0.005443,0.005436,279673.0,135,0.068173,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
3,4,client_23a62021009f63c4,content_44f34c0a90047651,1-3,0.000132,0.005443,0.005311,302868.0,65,0.067030,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
4,5,client_e547b89c05043229,content_306bc78dff1eb683,1-3,0.000504,0.005443,0.004939,242280.0,135,0.061239,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
5,6,client_23a62021009f63c4,content_2ac8c7995de53cd1,1-3,0.000191,0.005443,0.005252,94198.0,78,0.060151,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
6,7,client_e547b89c05043229,content_c46df0fa61530d86,1-3,0.000487,0.005443,0.004956,170379.0,135,0.059697,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
7,8,client_e547b89c05043229,content_35f10273f0baf406,1-3,0.000214,0.005443,0.005229,79593.0,135,0.059012,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
8,9,client_23a62021009f63c4,content_bf078007df823490,1-3,0.000000,0.005443,0.005443,44707.0,20,0.058283,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
9,10,client_23a62021009f63c4,content_66d3e7a515e4ec68,1-3,0.000132,0.005443,0.005311,53174.0,171,0.057795,HIGH_VISIBILITY_CTR_GAP,Review title and meta description


### Intended use and limits

This playbook is decision-support for prioritizing content review.

It identifies pages with a measured historical CTR opportunity relative to their position bucket baseline and ranks them for human review.

The scores are directional, not predictions of guaranteed traffic or clicks.

The playbook should be used to decide what to review first. It should not automatically rewrite, publish, delete, or redirect content.

The analysis is based on observed historical search performance from the training period and is validated against the later validation period. Results may become stale as search behavior, rankings, content, and site conditions change.

In [35]:
# Monitoring checks for recommendation staleness

print("Validation period:",
      VALID_START, "to", VALID_END)

print("\nPages evaluated:", len(model_dataset))

print("\nValidation CTR gap:")
print(model_dataset["ctr_gap"].describe())

print("\nOpportunity score:")
print(model_dataset["opportunity_score"].describe())

print("\nPositive validation CTR gaps:",
      (model_dataset["ctr_gap"] > 0).sum())

print("Zero/negative validation CTR gaps:",
      (model_dataset["ctr_gap"] <= 0).sum())

Validation period: 2026-04-01 to 2026-06-30

Pages evaluated: 104173

Validation CTR gap:
count    104173.000000
mean          0.000941
std           0.003541
min          -0.101081
25%          -0.000017
50%           0.001559
75%           0.003267
max           0.005443
Name: ctr_gap, dtype: float64

Opportunity score:
count    104173.000000
mean          0.012219
std           0.011333
min           0.000000
25%           0.000000
50%           0.010094
75%           0.020595
max           0.069787
Name: opportunity_score, dtype: float64

Positive validation CTR gaps: 77950
Zero/negative validation CTR gaps: 26223


In [36]:
# Check whether the highest-scored pages continue to outperform
# the lower-scored pages in validation

top_10_pct = model_dataset[
    model_dataset["opportunity_score"]
    >= model_dataset["opportunity_score"].quantile(0.90)
]

bottom_10_pct = model_dataset[
    model_dataset["opportunity_score"]
    <= model_dataset["opportunity_score"].quantile(0.10)
]

print("Top 10% pages:", len(top_10_pct))
print(
    "Top 10% average validation CTR gap:",
    top_10_pct["ctr_gap"].mean()
)

print("\nBottom 10% pages:", len(bottom_10_pct))
print(
    "Bottom 10% average validation CTR gap:",
    bottom_10_pct["ctr_gap"].mean()
)

Top 10% pages: 10418
Top 10% average validation CTR gap: 0.0033379639263265734

Bottom 10% pages: 29498
Bottom 10% average validation CTR gap: -0.0012721936533723896


In [37]:
# Restore the capstone time periods

TRAIN_START = "2025-10-01"
TRAIN_END = "2026-03-31"

VALID_START = "2026-04-01"
VALID_END = "2026-06-30"

print("Training period:", TRAIN_START, "to", TRAIN_END)
print("Validation period:", VALID_START, "to", VALID_END)

Training period: 2025-10-01 to 2026-03-31
Validation period: 2026-04-01 to 2026-06-30


In [38]:
# Section 2: Intended-use validation checks

print("Training period:", TRAIN_START, "to", TRAIN_END)
print("Validation period:", VALID_START, "to", VALID_END)

print("Model pages:", len(model_dataset))
print("Ranked action queue:", len(action_queue))

print("Opportunity score available:",
      "opportunity_score" in model_dataset.columns)

print("Validation CTR available:",
      "validation_ctr" in model_dataset.columns)

print("Historical CTR gap available:",
      "historical_ctr_gap" in model_dataset.columns)

Training period: 2025-10-01 to 2026-03-31
Validation period: 2026-04-01 to 2026-06-30
Model pages: 104173
Ranked action queue: 74675
Opportunity score available: True
Validation CTR available: True
Historical CTR gap available: True


### Human review

Every recommended action is decision-support, not an automatic instruction.

Before acting, a person should check:

- The page is still relevant to the business and search intent.
- The title and meta description accurately describe the page.
- The page has enough search visibility to justify the recommended action.
- Recent changes, seasonality, or unusual traffic patterns do not explain the observed CTR.
- The recommendation does not conflict with current SEO or editorial priorities.
- The proposed change can be tested and measured after publication.

### No-go list

Do not automate:

- Publishing title or meta description changes.
- Deleting or redirecting pages.
- Changing canonical URLs.
- Changing robots.txt or indexing directives.
- Making changes that could affect a site's legal, brand, or editorial requirements.
- Treating the opportunity score as proof that a change will improve performance.

The queue should be used to prioritize human review. It should not make irreversible website changes automatically.

In [39]:
# Section 3: Human review and no-go validation checks

print("Ranked action queue:", len(action_queue))

print(
    "Required review fields available:",
    all(
        col in action_queue.columns
        for col in [
            "client_hash_id",
            "content_hash_id",
            "position_bucket",
            "opportunity_score",
            "reason_code",
            "recommended_action",
        ]
    )
)

print(
    "No client names or URLs in queue:",
    not any(
        col.lower() in ["client_name", "url", "query"]
        for col in action_queue.columns
    )
)

print("Reason codes:", action_queue["reason_code"].nunique())
print("Recommended actions:", action_queue["recommended_action"].nunique())

Ranked action queue: 74675
Required review fields available: True
No client names or URLs in queue: True
Reason codes: 3
Recommended actions: 3


### Monitoring and retrain triggers

The recommendations should be monitored against new validation data.

Retraining or recalibration should be considered when:

- The relationship between opportunity score and observed CTR gap weakens.
- The ranking of high-score pages stops identifying larger observed CTR gaps.
- The validation CTR gap distribution changes materially.
- Search performance or the available data coverage changes.
- The training data becomes stale relative to the period being evaluated.

The score is directional decision-support. It should be rechecked on new data rather than treated as a permanent ranking.

### Monitoring / retrain triggers

The opportunity score is intended for decision-support, not automatic content changes.

In the validation period, higher score deciles showed progressively larger observed CTR gaps. The decile-to-validation-gap correlation was 0.933, which provides directional evidence that the ranking signal remained useful on later data.

Monitor the relationship between opportunity score and observed CTR gap over time. Review the model if this relationship weakens materially, the score distribution changes substantially, or validation coverage becomes too sparse to support the recommendations.

Retrain or revisit the scoring rules when the recommendations no longer show a useful relationship with observed outcomes.

In [40]:
# Section 4: Monitoring / retrain validation checks

# Align validation gap direction with the historical opportunity definition.
# Positive values mean observed CTR was below the position baseline.

model_dataset["validation_ctr_gap"] = (
    model_dataset["baseline_ctr"]
    - model_dataset["validation_ctr"]
)

decile_check = (
    model_dataset
    .groupby("score_decile", observed=True)
    .agg(
        pages=("opportunity_score", "size"),
        avg_score=("opportunity_score", "mean"),
        avg_validation_ctr_gap=("validation_ctr_gap", "mean"),
        total_validation_impressions=("validation_impressions", "sum")
    )
    .reset_index()
)

print("Score decile monitoring:")
display(decile_check)

correlation = decile_check["score_decile"].corr(
    decile_check["avg_validation_ctr_gap"]
)

print("\nDecile-to-validation-gap correlation:", correlation)

Score decile monitoring:


,score_decile,pages,avg_score,avg_validation_ctr_gap,total_validation_impressions
0,0,31252,0.000036,-0.001159,254964155.0
1,1,10417,0.004803,0.000915,63770977.0
2,2,10418,0.008836,0.000955,34874032.0
3,3,10417,0.012050,0.001470,60741481.0
4,4,10417,0.016922,0.001772,47879561.0
5,5,10417,0.020607,0.002156,46017383.0
6,6,10417,0.024551,0.002283,69910377.0
7,7,10418,0.034308,0.003338,87491159.0



Decile-to-validation-gap correlation: 0.9327716423818803


### Results

The analysis evaluated 104,173 pages that appeared in both the training and validation periods.

The opportunity score was built from historical CTR performance relative to the position-bucket baseline, then evaluated on the later validation period.

The highest opportunity-score group showed an average validation CTR gap of 0.003338, while the lowest-score group showed -0.001159.

The difference between these groups was 0.004497 CTR points.

The decile-to-validation-gap correlation was 0.933, providing directional evidence that higher opportunity scores were associated with larger observed CTR gaps in the later period.

This result supports the score as a prioritization signal for human review. It does not establish that changing a page will cause a CTR increase.

In [41]:
# Section 4: Results vs baseline

top_10_pct = model_dataset[
    model_dataset["opportunity_score"]
    >= model_dataset["opportunity_score"].quantile(0.90)
]

bottom_10_pct = model_dataset[
    model_dataset["opportunity_score"]
    <= model_dataset["opportunity_score"].quantile(0.10)
]

top_gap = top_10_pct["ctr_gap"].mean()
bottom_gap = bottom_10_pct["ctr_gap"].mean()
absolute_difference = top_gap - bottom_gap

results_table = pd.DataFrame({
    "group": ["Bottom 10% score", "Top 10% score"],
    "pages": [len(bottom_10_pct), len(top_10_pct)],
    "avg_validation_ctr_gap": [bottom_gap, top_gap]
})

print("Results vs baseline:")
display(results_table)

print("Difference between top and bottom groups:", absolute_difference)

Results vs baseline:


,group,pages,avg_validation_ctr_gap
0,Bottom 10% score,29498,-0.001272
1,Top 10% score,10418,0.003338


Difference between top and bottom groups: 0.004610157579698963


### Exports for the paper

The ranked action queue is exported for use in the research paper.

The export contains only hashed client and content identifiers, ranking information, reason codes, recommended actions, and supporting measurements. It does not contain client names, URLs, or private search queries.

In [42]:
# Section 5: Export the action queue for the paper

from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/FlyRank-ml-Internship-/work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Export the ranked action queue
queue_path = OUTPUT_DIR / "content_action_queue.csv"
action_queue.to_csv(queue_path, index=False)

# Export the monitoring table
monitoring_path = OUTPUT_DIR / "score_decile_monitoring.csv"
decile_check.to_csv(monitoring_path, index=False)

print("Action queue exported:", queue_path)
print("Monitoring table exported:", monitoring_path)
print("Queue rows:", len(action_queue))
print("Monitoring rows:", len(decile_check))

Action queue exported: /content/drive/MyDrive/FlyRank-ml-Internship-/work/outputs/content_action_queue.csv
Monitoring table exported: /content/drive/MyDrive/FlyRank-ml-Internship-/work/outputs/score_decile_monitoring.csv
Queue rows: 74675
Monitoring rows: 8


## 5. Limitations

This analysis measures observed associations in historical search data. It does not prove that an optimization will cause higher CTR.

The position-bucket baseline is intentionally simple and does not capture every factor that can affect CTR.

The validation period is later than the training period, which provides a temporal check but does not replace an experiment.

Some pages have limited validation activity, so their observed CTR can be noisy.

The opportunity score should not be interpreted as a forecast of guaranteed clicks, traffic, rankings, or revenue.

The recommendations also do not inspect the actual page copy, title, meta description, search query, business context, or editorial requirements. A human must review those factors before making changes.

In [43]:
# Section 5: Limitation checks

print("Pages evaluated:", len(model_dataset))
print("Training period:", TRAIN_START, "to", TRAIN_END)
print("Validation period:", VALID_START, "to", VALID_END)

print(
    "Validation impressions available:",
    model_dataset["validation_impressions"].notna().sum()
)

print(
    "Pages with zero validation impressions:",
    (model_dataset["validation_impressions"] == 0).sum()
)

print(
    "Opportunity score range:",
    model_dataset["opportunity_score"].min(),
    "to",
    model_dataset["opportunity_score"].max()
)

Pages evaluated: 104173
Training period: 2025-10-01 to 2026-03-31
Validation period: 2026-04-01 to 2026-06-30
Validation impressions available: 104173
Pages with zero validation impressions: 0
Opportunity score range: 0.0 to 0.06978658996419819


## 6. Ranked recommendations

Prioritize pages with the highest opportunity scores for human review.

The first review group is pages with high visibility and a large historical CTR gap relative to their position bucket. These pages receive the highest priority because they combine observed underperformance with meaningful search visibility.

For pages ranking in positions 1–5, review the title and meta description first.

For pages ranking in positions 6–20, review the snippet, title, meta description, content relevance, and alignment with search intent.

For pages ranking 21+, review whether the page is targeting the right search intent and whether the content provides a strong reason to compete before making CTR-focused changes.

The ranked queue contains 74,675 pages. The queue is a prioritization tool for human review, not an automated publishing system.

In [44]:
# Section 6: Ranked recommendations

print("Total ranked recommendations:", len(action_queue))

print("\nReason code distribution:")
display(
    action_queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="pages")
)

print("\nRecommended action distribution:")
display(
    action_queue["recommended_action"]
    .value_counts()
    .rename_axis("recommended_action")
    .reset_index(name="pages")
)

print("\nTop 20 recommendations:")
display(action_queue.head(20))

Total ranked recommendations: 74675

Reason code distribution:


,reason_code,pages
0,LOW_VISIBILITY_CTR_GAP,30837
1,MID_VISIBILITY_CTR_GAP,22279
2,HIGH_VISIBILITY_CTR_GAP,21559



Recommended action distribution:


,recommended_action,pages
0,Review content relevance and search intent,30837
1,Review snippet and on-page relevance,22279
2,Review title and meta description,21559



Top 20 recommendations:


,rank,client_hash_id,content_hash_id,position_bucket,ctr,baseline_ctr,historical_ctr_gap,impressions,active_days,opportunity_score,reason_code,recommended_action
0,1,client_73cda7b4e4f265ea,content_fec55986a1868d62,1-3,0.000003,0.005443,0.005440,372381.0,182,0.069787,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,1-3,0.000011,0.005443,0.005432,366197.0,182,0.069590,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
2,3,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,1-3,0.000007,0.005443,0.005436,279673.0,135,0.068173,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
3,4,client_23a62021009f63c4,content_44f34c0a90047651,1-3,0.000132,0.005443,0.005311,302868.0,65,0.067030,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
4,5,client_e547b89c05043229,content_306bc78dff1eb683,1-3,0.000504,0.005443,0.004939,242280.0,135,0.061239,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
5,6,client_23a62021009f63c4,content_2ac8c7995de53cd1,1-3,0.000191,0.005443,0.005252,94198.0,78,0.060151,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
6,7,client_e547b89c05043229,content_c46df0fa61530d86,1-3,0.000487,0.005443,0.004956,170379.0,135,0.059697,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
7,8,client_e547b89c05043229,content_35f10273f0baf406,1-3,0.000214,0.005443,0.005229,79593.0,135,0.059012,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
8,9,client_23a62021009f63c4,content_bf078007df823490,1-3,0.000000,0.005443,0.005443,44707.0,20,0.058283,HIGH_VISIBILITY_CTR_GAP,Review title and meta description
9,10,client_23a62021009f63c4,content_66d3e7a515e4ec68,1-3,0.000132,0.005443,0.005311,53174.0,171,0.057795,HIGH_VISIBILITY_CTR_GAP,Review title and meta description


## 7. Artifacts the paper embeds

The analysis produces two reusable artifacts.

1. `content_action_queue.csv` contains the ranked action queue with hashed identifiers, position buckets, opportunity scores, reason codes, recommended actions, and supporting measurements.

2. `score_decile_monitoring.csv` contains the score-decile validation results used to monitor whether higher opportunity scores continue to correspond with larger observed CTR gaps.

These artifacts support the deployed research paper and make the recommendations reproducible from the notebook outputs.

In [45]:
# Section 7: Verify paper artifacts

from pathlib import Path

print("Action queue rows:", len(action_queue))
print("Monitoring rows:", len(decile_check))

print("\nAction queue columns:")
print(list(action_queue.columns))

print("\nMonitoring columns:")
print(list(decile_check.columns))

print("\nArtifacts:")
print(queue_path)
print(monitoring_path)

print("\nFiles exist:")
print("Action queue:", queue_path.exists())
print("Monitoring table:", monitoring_path.exists())

Action queue rows: 74675
Monitoring rows: 8

Action queue columns:
['rank', 'client_hash_id', 'content_hash_id', 'position_bucket', 'ctr', 'baseline_ctr', 'historical_ctr_gap', 'impressions', 'active_days', 'opportunity_score', 'reason_code', 'recommended_action']

Monitoring columns:
['score_decile', 'pages', 'avg_score', 'avg_validation_ctr_gap', 'total_validation_impressions']

Artifacts:
/content/drive/MyDrive/FlyRank-ml-Internship-/work/outputs/content_action_queue.csv
/content/drive/MyDrive/FlyRank-ml-Internship-/work/outputs/score_decile_monitoring.csv

Files exist:
Action queue: True
Monitoring table: True


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [46]:
# Section 4: Results vs baseline

top_decile = (
    model_dataset[model_dataset["score_decile"] == model_dataset["score_decile"].max()]
)

bottom_decile = (
    model_dataset[model_dataset["score_decile"] == model_dataset["score_decile"].min()]
)

top_gap = top_decile["validation_ctr_gap"].mean()
bottom_gap = bottom_decile["validation_ctr_gap"].mean()

results_table = pd.DataFrame({
    "group": ["Lowest score decile", "Highest score decile"],
    "pages": [len(bottom_decile), len(top_decile)],
    "avg_validation_ctr_gap": [bottom_gap, top_gap],
    "total_validation_impressions": [
        bottom_decile["validation_impressions"].sum(),
        top_decile["validation_impressions"].sum()
    ]
})

display(results_table)

print("Absolute difference:", top_gap - bottom_gap)
print(
    "Decile-to-validation-gap correlation:",
    model_dataset["score_decile"].corr(
        model_dataset["validation_ctr_gap"]
    )
)

,group,pages,avg_validation_ctr_gap,total_validation_impressions
0,Lowest score decile,31252,-0.001159,254964155.0
1,Highest score decile,10418,0.003338,87491159.0


Absolute difference: 0.004497307387131298
Decile-to-validation-gap correlation: 0.4075621163492313


## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
